# Dataset clima

Este codigo descraga los datos metereológicos de Chicago desde Open-Meteo.

La API de datos meteorológicos históricos se basa en conjuntos de datos de reanálisis y utiliza una combinación de observaciones de estaciones meteorológicas, aviones, boyas, radares y satélites para crear un registro exhaustivo de las condiciones meteorológicas del pasado. Estos conjuntos de datos permiten subsanar las lagunas mediante el uso de modelos matemáticos para estimar los valores de diversas variables meteorológicas. Como resultado, los conjuntos de datos de reanálisis pueden proporcionar información meteorológica histórica detallada de lugares que quizá no contaran con estaciones meteorológicas cercanas, como las zonas rurales o el océano abierto.

Los modelos para los datos meteorológicos históricos utilizan una resolución espacial de 9 km para captar detalles precisos cerca de las costas o en terrenos montañosos complejos. En general, una mayor resolución espacial significa que los datos son más detallados y representan las condiciones meteorológicas con mayor precisión a escalas más pequeñas.

El conjunto de datos IFS del ECMWF ha sido recopilado meticulosamente por Open-Meteo utilizando simulaciones diarias a las 0z, 6z, 12z y 18z, empleando la versión más actualizada del IFS. Este conjunto de datos ofrece la máxima resolución y precisión para las condiciones meteorológicas históricas a escala global.

**Variables:**

| Variable | Periodo | Unidad | Descripción |
|----------|------------|------|-------------|
| `temperature_2m` | Instantáneo | °C | Temperatura del aire a 2 metros sobre el suelo |
| `relative_humidity_2m` | Instantáneo | % | Humedad relativa a 2 metros sobre el suelo |
| `rain` | Suma de la hora anterior | mm | Las precipitaciones líquidas de la hora anterior, incluidas los chubascos locales y la lluvia procedente de sistemas a gran escala |
|`snowfall` | Suma de la hora anterior | cm | Cantidad de nieve caída durante la hora anterior  |
| `wind_speed_10m` | Instantáneo | km/h | Velocidad del viento a 10 metros sobre el suelo |

In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.85,
	"longitude": -87.65,
	"start_date": "2011-01-01",
	"end_date": "2026-07-31",
	"hourly": ["temperature_2m", "rain", "snowfall", "relative_humidity_2m", "wind_speed_10m"],
	"timezone": "GMT",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(2).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 41.8629150390625°N -87.64877319335938°E
Elevation: 179.0 m asl
Timezone: NoneNone
Timezone difference to GMT+0: 0s

Hourly data
                             date  temperature_2m  rain  snowfall  \
0      2011-01-01 00:00:00+00:00        9.050000   0.0       0.0   
1      2011-01-01 01:00:00+00:00        9.050000   0.0       0.0   
2      2011-01-01 02:00:00+00:00        9.050000   0.2       0.0   
3      2011-01-01 03:00:00+00:00        9.150000   0.1       0.0   
4      2011-01-01 04:00:00+00:00        9.100000   0.0       0.0   
...                          ...             ...   ...       ...   
136579 2026-07-31 19:00:00+00:00       25.700001   0.3       0.0   
136580 2026-07-31 20:00:00+00:00       23.750000   1.9       0.0   
136581 2026-07-31 21:00:00+00:00       23.400000   0.9       0.0   
136582 2026-07-31 22:00:00+00:00       22.750000   1.4       0.0   
136583 2026-07-31 23:00:00+00:00       23.100000   0.2       0.0   

        relative_humidity_2m  wind_speed_

In [2]:
# lo convertimos en csv

hourly_dataframe.to_csv("data_clean/weather_data.csv", index=False, encoding='utf-8')